Notes: 

* Simple natural language model DistilBERT is used (simpler version of BERT (transformers))

* We ignore inputs "keyword" and "location", making the model text-only.

In [1]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

# --- 1. LOAD THE DATA ---
# Assuming you downloaded 'train.csv' from Kaggle and uploaded it to Colab
# The Kaggle dataset has columns: 'text' (the tweet) and 'target' (1 for disaster, 0 for not)
df = pd.read_csv('train.csv')
df = df[['text', 'target']] # Keep only what we need
df = df.dropna()            # Remove empty rows

# ---> THE FIX: Rename 'target' to 'label' <--- THE TRAINER'S DEFAULT IS "label" when comparing to correct class/target
df = df.rename(columns={'target': 'label'})

# Convert pandas dataframe to Hugging Face Dataset format
hf_dataset = Dataset.from_pandas(df)
# Split into 80% training and 20% testing
hf_dataset = hf_dataset.train_test_split(test_size=0.2) 

# --- 2. THE TOKENIZER (Text to Numbers) ---
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # padding="max_length" makes all sentences the same length by adding zeros
    # truncation=True cuts off sentences that are too long
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Apply the tokenizer to our entire dataset
tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)

# --- 3. THE MODEL ---
# Load DistilBERT and tell it we are doing Binary Classification (num_labels=2)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
#model = AutoModel.from_pretrained(model_name, num_labels=2)

# Define how we measure success (Accuracy)
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# --- 4. TRAINING SETUP ---
training_args = TrainingArguments(
    eval_strategy="epoch", # Check accuracy at the end of each epoch
    learning_rate=2e-5,          # Standard learning rate for fine-tuning
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,          # 3 passes over the data is usually enough
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

# --- 5. TRAIN IT! ---
print("Starting training...")
trainer.train()

# --- 6. TEST IT ON NEW TWEETS ---
def predict_tweet(tweet):
    inputs = tokenizer(tweet, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get the predicted class (0 or 1)
    prediction = torch.argmax(outputs.logits, dim=-1).item()
    confidence = torch.softmax(outputs.logits, dim=-1).max().item()
    
    label = "Disaster" if prediction == 1 else "Not a Disaster"
    print(f"Tweet: '{tweet}' \nPrediction: {label} (Confidence: {confidence*100:.1f}%)\n")

print("\n--- Testing Custom Inputs ---")
predict_tweet("Just felt a huge earthquake in San Francisco! Buildings are shaking!")
predict_tweet("The new Godzilla movie is a total disaster, horrible acting.")

Map:   0%|          | 0/6090 [00:00<?, ? examples/s]

Map:   0%|          | 0/1523 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training...


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## Save model with weights to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 1. Define the exact folder path on your Google Drive
# Hugging Face will create this folder for you if it doesn't exist!
save_directory = "/content/drive/My Drive/text_branch"

# 2. Save the Model (Weights + Config)
trainer.save_model(save_directory)

# 3. Save the Tokenizer (Crucial Best Practice!)
# You always want to save the exact tokenizer rules with the model
tokenizer.save_pretrained(save_directory)

print(f"Success! Model and tokenizer safely saved to: {save_directory}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Success! Model and tokenizer safely saved to: /content/drive/My Drive/text_branch
